In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

## Load Models & Build Comparison Table

In [42]:
# Reload DBSCAN Data
dbscan_df = pd.read_csv("../Data/generated/trips_dbscan_labeled.csv")

dbscan_df["pickup_lat"]     = pd.to_numeric(dbscan_df["pickup_lat"],     errors="coerce")
dbscan_df["pickup_lon"]     = pd.to_numeric(dbscan_df["pickup_lon"],     errors="coerce")
dbscan_df["hour_sin"]       = pd.to_numeric(dbscan_df["hour_sin"],       errors="coerce")
dbscan_df["hour_cos"]       = pd.to_numeric(dbscan_df["hour_cos"],       errors="coerce")
dbscan_df["co2TailpipeGpm"] = pd.to_numeric(dbscan_df["co2TailpipeGpm"], errors="coerce")
dbscan_df["co2_total_g"]    = pd.to_numeric(dbscan_df["co2_total_g"],    errors="coerce")
dbscan_df["trip_miles"]     = pd.to_numeric(dbscan_df["trip_miles"],     errors="coerce")

dbscan_n_clusters = dbscan_df["dbscan_cluster"].nunique() - (1 if -1 in dbscan_df["dbscan_cluster"].values else 0)
dbscan_noise_pct  = (dbscan_df["dbscan_cluster"] == -1).mean() * 100

mask_db  = dbscan_df["dbscan_cluster"] != -1
X_db     = StandardScaler().fit_transform(
    dbscan_df.loc[mask_db, ["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]].values
)
sil_db   = silhouette_score(X_db, dbscan_df.loc[mask_db, "dbscan_cluster"],
                             sample_size=min(5000, mask_db.sum()), random_state=RANDOM_STATE)
dbi_db   = davies_bouldin_score(X_db, dbscan_df.loc[mask_db, "dbscan_cluster"])


# Reload K-Means Data
kmeans_df = pd.read_csv("../Data/generated/trips_kmeans_labeled.csv") 

kmeans_df["pickup_lat"]     = pd.to_numeric(kmeans_df["pickup_lat"], errors="coerce")
kmeans_df["pickup_lon"]     = pd.to_numeric(kmeans_df["pickup_lon"], errors="coerce")
kmeans_df["hour_sin"]       = pd.to_numeric(kmeans_df["hour_sin"], errors="coerce")
kmeans_df["hour_cos"]       = pd.to_numeric(kmeans_df["hour_cos"], errors="coerce")
kmeans_df["co2TailpipeGpm"] = pd.to_numeric(kmeans_df["co2TailpipeGpm"], errors="coerce")
kmeans_df["co2_total_g"]    = pd.to_numeric(kmeans_df["co2_total_g"], errors="coerce")
kmeans_df["trip_miles"]     = pd.to_numeric(kmeans_df["trip_miles"], errors="coerce")

# Extract cluster labels
km_labels = kmeans_df["kmeans_cluster"] 

# Standardize features for metrics
mask_km = ~km_labels.isna()  # just in case there are missing labels
X_km    = StandardScaler().fit_transform(
    kmeans_df.loc[mask_km, ["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]].values
)

# Metrics
sil_km = silhouette_score(X_km, km_labels[mask_km],
                          sample_size=min(5000, mask_km.sum()), random_state=RANDOM_STATE)
dbi_km = davies_bouldin_score(X_km, km_labels[mask_km])
K = km_labels.nunique()

cluster_sizes = (
    kmeans_df.groupby("kmeans_cluster").size()
    .reset_index(name="n_trips")
    .sort_values("n_trips", ascending=False)
    .reset_index(drop=True)
)

# Reload Hierarchical Clustering Data
hier_df = pd.read_csv("../Data/generated/trips_hierarchical_labeled.csv")

# Make sure numeric columns are correct
hier_df["pickup_lat"]   = pd.to_numeric(hier_df["pickup_lat"], errors="coerce")
hier_df["pickup_lon"]   = pd.to_numeric(hier_df["pickup_lon"], errors="coerce")
hier_df["hour_sin"]     = pd.to_numeric(hier_df["hour_sin"], errors="coerce")
hier_df["hour_cos"]     = pd.to_numeric(hier_df["hour_cos"], errors="coerce")
hier_df["co2_total_g"]  = pd.to_numeric(hier_df["co2_total_g"], errors="coerce")
hier_df["trip_miles"]   = pd.to_numeric(hier_df["trip_miles"], errors="coerce")

# Compute hierarchical cluster metrics
hier_n_clusters = hier_df["hier_cluster"].nunique()
mask_hier = hier_df["hier_cluster"] >= 0  # all rows are valid
X_hier = StandardScaler().fit_transform(hier_df.loc[mask_hier, ["pickup_lat","pickup_lon","hour_sin","hour_cos"]])

sil_hier = silhouette_score(X_hier, hier_df.loc[mask_hier, "hier_cluster"],
                             sample_size=min(5000, mask_hier.sum()), random_state=RANDOM_STATE)
dbi_hier = davies_bouldin_score(X_hier, hier_df.loc[mask_hier, "hier_cluster"])
cluster_sizes_hier = hier_df.groupby("hier_cluster").size()

comparison_df = pd.DataFrame([
    {
        "algorithm":      "DBSCAN",
        "n_clusters":      dbscan_n_clusters,
        "noise_pct":       round(dbscan_noise_pct, 1),
        "silhouette":      round(sil_db, 4),
        "davies_bouldin":  round(dbi_db, 4),
        "cluster_size_std": round(dbscan_df[dbscan_df["dbscan_cluster"]>=0].groupby("dbscan_cluster").size().std(), 1),
    },
    {
        "algorithm":      "K-Means",
        "n_clusters":      K,
        "noise_pct":       0.0,
        "silhouette":      round(sil_km, 4),
        "davies_bouldin":  round(dbi_km, 4),
        "cluster_size_std": round(cluster_sizes["n_trips"].std(), 1),
    },
    {
        "algorithm":      "Hierarchical",
        "n_clusters":      hier_n_clusters,
        "noise_pct":       0.0,
        "silhouette":      round(sil_hier, 4),
        "davies_bouldin":  round(dbi_hier, 4),
        "cluster_size_std": round(cluster_sizes_hier.std(), 1),
    },
])

print(comparison_df.to_string(index=False))
print("\nNotes:")
print("  silhouette    : higher is better (max 1.0)")
print("  davies_bouldin: lower is better")
print("  cluster_size_std: lower = more balanced cluster sizes")
print("  noise_pct     : DBSCAN labels outliers as noise; K-Means & Hierarchical assign all points")

   algorithm  n_clusters  noise_pct  silhouette  davies_bouldin  cluster_size_std
      DBSCAN          80       15.8     -0.1192          2.1875             160.9
     K-Means          16        0.0      0.2950          1.0420             306.9
Hierarchical          17        0.0      0.2396          1.1100             302.0

Notes:
  silhouette    : higher is better (max 1.0)
  davies_bouldin: lower is better
  cluster_size_std: lower = more balanced cluster sizes
  noise_pct     : DBSCAN labels outliers as noise; K-Means & Hierarchical assign all points


In [43]:
comparison_df.to_csv("../Data/generated/clustering_comparison.csv", index=False)
print(f"Saved clustering_comparison.csv    ({len(comparison_df)} algorithms)")

Saved clustering_comparison.csv    (3 algorithms)
